<a href="https://colab.research.google.com/github/aliftffd/AMC_notebook/blob/main/Improved_CNN2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torchinfo?

Object `torchinfo` not found.


In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import h5py
import json
from matplotlib import pyplot as plt
import torch
from torch import nn
from torchinfo import summary
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix,classification_report
from tqdm import trange, tqdm
import seaborn as sns

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("pinxau1000/radioml2018")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/radioml2018


In [5]:
n_channels=2
batch_size=512
frame_size = 4096
n_labels = 19 # adjust how many modulation need to train
# Number of frames per snr/modulation combination for train,valid and test data
nf_train = 1024
nf_valid = 512
nf_test = 256

In [6]:
def dataset_split(data,
                  modulations_classes,
                  modulations,
                  snrs,
                  target_modulations,
                  mode,
                  target_snrs,
                  train_proportion=0.7, #set 70% from dataset as train
                  valid_proportion=0.2, #Set 20 % from dataset as valid
                  test_proportion=0.1, # set 10 % as tetsing
                  seed=48):
    np.random.seed(seed)
    X_output = []
    Y_output = []
    Z_output = []

    target_modulation_indices = [modulations_classes.index(modu) for modu in target_modulations]

    for modu in target_modulation_indices:
        for snr in target_snrs:
            snr_modu_indices = np.where((modulations == modu) & (snrs == snr))[0]

            np.random.shuffle(snr_modu_indices)
            num_samples = len(snr_modu_indices)
            train_end = int(train_proportion * num_samples)
            valid_end = int((train_proportion + valid_proportion) * num_samples)

            if mode == 'train':
                indices = snr_modu_indices[:train_end]
            elif mode == 'valid':
                indices = snr_modu_indices[train_end:valid_end]
            elif mode == 'test':
                indices = snr_modu_indices[valid_end:]
            else:
                raise ValueError(f'unknown mode: {mode}. Valid modes are train, valid and test')

            X_output.append(data[np.sort(indices)])
            Y_output.append(modulations[np.sort(indices)])
            Z_output.append(snrs[np.sort(indices)])

    X_array = np.vstack(X_output)
    Y_array = np.concatenate(Y_output)
    Z_array = np.concatenate(Z_output)
    for index, value in enumerate(np.unique(np.copy(Y_array))):
        Y_array[Y_array == value] = index
    return X_array, Y_array, Z_array

In [7]:
class RadioML18Dataset(Dataset):
    def __init__(self, mode: str,seed=48,):
        super(RadioML18Dataset, self).__init__()

        # load data
        hdf5_file = h5py.File("/root/.cache/kagglehub/datasets/pinxau1000/radioml2018/versions/2/GOLD_XYZ_OSC.0001_1024.hdf5",  'r')
        self.modulation_classes = json.load(open("/root/.cache/kagglehub/datasets/pinxau1000/radioml2018/versions/2/classes-fixed.json", 'r'))
        self.X = hdf5_file['X']
        self.Y = np.argmax(hdf5_file['Y'], axis=1)
        self.Z = hdf5_file['Z'][:, 0]

        train_proportion=(24*26*nf_train)/self.X.shape[0]
        valid_proportion=(24*26*nf_valid)/self.X.shape[0]
        test_proportion=(24*26*nf_test)/self.X.shape[0]

        """target_modulations =['OOK', '4ASK', 'BPSK', 'QPSK', '8PSK',
        '16QAM', 'AM-SSB-SC', 'AM-DSB-SC', 'FM', 'GMSK','OQPSK']target
        modulation class and snr"""

        # in this line i could change it the target modulation
        self.target_modulations = ['OOK', '4ASK', '8ASK', 'BPSK', 'QPSK', '8PSK', '16PSK', '32PSK',
                           '16APSK', '32APSK', '64APSK', '128APSK', '16QAM', '32QAM',
                           '64QAM', '128QAM', '256QAM', 'GMSK', 'OQPSK']


        self.target_snrs = np.unique(self.Z)

        self.X_data, self.Y_data, self.Z_data = dataset_split(
                                                                  data = self.X,
                                                                  modulations_classes = self.modulation_classes,
                                                                  modulations = self.Y,
                                                                  snrs = self.Z,
                                                                  mode = mode,
                                                                  train_proportion = train_proportion,
                                                                  valid_proportion = valid_proportion,
                                                                  test_proportion = test_proportion,
                                                                  target_modulations = self.target_modulations,
                                                                  target_snrs  = self.target_snrs,
                                                                  seed=48
                                                                 )

        # store statistic of whole dataset
        self.num_data = self.X_data.shape[0]
        self.num_lbl = len(self.target_modulations)
        self.num_snr = self.target_snrs.shape[0]

    def __len__(self):
        return self.X_data.shape[0]

    def __getitem__(self, idx):
        x,y,z = self.X_data[idx], self.Y_data[idx], self.Z_data[idx]
        x,y,z = torch.Tensor(x).transpose(0, 1) , y , z
        return x,y,z

In [8]:
ds = RadioML18Dataset(mode='test')
data_len = ds.num_data
n_labels=ds.num_lbl
n_snrs = ds.num_snr
frame_size=ds.X.shape[1]

del ds

In [9]:
dataset = RadioML18Dataset(mode='train')

# Print all modulation classes
print("All Modulation Classes:", dataset.modulation_classes)

All Modulation Classes: ['OOK', '4ASK', '8ASK', 'BPSK', 'QPSK', '8PSK', '16PSK', '32PSK', '16APSK', '32APSK', '64APSK', '128APSK', '16QAM', '32QAM', '64QAM', '128QAM', '256QAM', 'AM-SSB-WC', 'AM-SSB-SC', 'AM-DSB-WC', 'AM-DSB-SC', 'FM', 'GMSK', 'OQPSK']


In [10]:
import time
st = time.time()
train_dl = DataLoader(dataset=RadioML18Dataset(mode='train'),batch_size = 64, shuffle = True, drop_last = True)
valid_dl = DataLoader(dataset=RadioML18Dataset(mode='valid'),batch_size = 128, shuffle = True, drop_last = False)
test_dl = DataLoader(dataset=RadioML18Dataset(mode='train'), batch_size = 128, shuffle = True, drop_last = False)
et = time.time()
elapsed_time = et - st
print(f'Execution time : {elapsed_time} second')

Execution time : 40.739184617996216 second


In [11]:
class CNN_Block(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.25)
        )

    def forward(self, x):
        return self.block(x)

# Define the full CNN network
class CNN_NET(nn.Module):
    def __init__(self, n_labels):
        super().__init__()
        self.backbone = nn.Sequential(
            CNN_Block(2, 32),
            CNN_Block(32, 64),
            CNN_Block(64, 128),
            CNN_Block(128, 128),
            nn.AdaptiveAvgPool1d(8)  # Fixed-size output
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, n_labels)
        )

    def forward(self, x):
        x = self.backbone(x)
        return self.classifier(x)

In [12]:
class ImprovedCNN_Block(nn.Module):
    def __init__(self, in_channels, out_channels, dropout_rate=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(dropout_rate)
        )

    def forward(self, x):
        return self.block(x)

class ImprovedCNN_NET(nn.Module):
    def __init__(self, n_labels, dropout_rate=0.4):
        super().__init__()

        self.backbone = nn.Sequential(
            ImprovedCNN_Block(2, 32, dropout_rate=0.2),
            ImprovedCNN_Block(32, 64, dropout_rate=0.3),
            ImprovedCNN_Block(64, 128, dropout_rate=0.3),
            nn.AdaptiveAvgPool1d(16)
        )

        # Enhanced classifier for 19-class problem
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate * 0.75),

            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate * 0.5),

            nn.Linear(128, n_labels)  # This will be 19 for full dataset
        )

    def forward(self, x):
        x = self.backbone(x)
        return self.classifier(x)

def create_improved_model(n_labels, dropout_rate=0.4):
    """Create improved CNN model - now works for any number of classes"""
    return ImprovedCNN_NET(n_labels, dropout_rate)

In [13]:
class EarlyStopping:
    def __init__(self, patience=15, min_delta=0.001, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_loss = float('inf')
        self.counter = 0
        self.best_weights = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            if self.restore_best_weights:
                self.best_weights = model.state_dict().copy()
        else:
            self.counter += 1

        if self.counter >= self.patience:
            if self.restore_best_weights and self.best_weights is not None:
                model.load_state_dict(self.best_weights)
            return True
        return False

In [14]:
model =CNN_NET(n_labels).to('cuda')

# Safer: create explicit dummy input
dummy_input = torch.randn(1, n_channels, frame_size).to('cuda')

summary(model, input_data=dummy_input)

Layer (type:depth-idx)                   Output Shape              Param #
CNN_NET                                  [1, 19]                   --
├─Sequential: 1-1                        [1, 128, 8]               --
│    └─CNN_Block: 2-1                    [1, 32, 512]              --
│    │    └─Sequential: 3-1              [1, 32, 512]              288
│    └─CNN_Block: 2-2                    [1, 64, 256]              --
│    │    └─Sequential: 3-2              [1, 64, 256]              6,336
│    └─CNN_Block: 2-3                    [1, 128, 128]             --
│    │    └─Sequential: 3-3              [1, 128, 128]             24,960
│    └─CNN_Block: 2-4                    [1, 128, 64]              --
│    │    └─Sequential: 3-4              [1, 128, 64]              49,536
│    └─AdaptiveAvgPool1d: 2-5            [1, 128, 8]               --
├─Sequential: 1-2                        [1, 19]                   --
│    └─Flatten: 2-6                      [1, 1024]                 --
│  

In [15]:
model =ImprovedCNN_NET(n_labels).to('cuda')

# Safer: create explicit dummy input
dummy_input = torch.randn(1, n_channels, frame_size).to('cuda')

summary(model, input_data=dummy_input)

Layer (type:depth-idx)                   Output Shape              Param #
ImprovedCNN_NET                          [1, 19]                   --
├─Sequential: 1-1                        [1, 128, 16]              --
│    └─ImprovedCNN_Block: 2-1            [1, 32, 512]              --
│    │    └─Sequential: 3-1              [1, 32, 512]              3,456
│    └─ImprovedCNN_Block: 2-2            [1, 64, 256]              --
│    │    └─Sequential: 3-2              [1, 64, 256]              18,816
│    └─ImprovedCNN_Block: 2-3            [1, 128, 128]             --
│    │    └─Sequential: 3-3              [1, 128, 128]             74,496
│    └─AdaptiveAvgPool1d: 2-4            [1, 128, 16]              --
├─Sequential: 1-2                        [1, 19]                   --
│    └─Flatten: 2-5                      [1, 2048]                 --
│    └─Linear: 2-6                       [1, 512]                  1,049,088
│    └─BatchNorm1d: 2-7                  [1, 512]                  

In [16]:
# ============================================================================
# 1. IMPROVED TRAINING FUNCTION
# ============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
import gc
from tqdm import tqdm, trange
import numpy as np
from collections import deque


def train_model(model, train_dl, valid_dl, verbose=True, device='cuda', num_epoch=200,
                accumulation_steps=1, use_amp=True, memory_cleanup_freq=10):
    """
    GPU memory-optimized training function for Google Colab Pro

    Args:
        model: Neural network model
        train_dl: Training dataloader
        valid_dl: Validation dataloader
        verbose: Print training progress
        device: Device to train on
        num_epoch: Maximum number of epochs
        accumulation_steps: Gradient accumulation steps (effective batch size = batch_size * accumulation_steps)
        use_amp: Use Automatic Mixed Precision (saves ~40-50% GPU memory)
        memory_cleanup_freq: How often to clean GPU memory (every N epochs)
    """

    # GPU memory check and optimization
    if device == 'cuda':
        print(f"GPU: {torch.cuda.get_device_name()}")
        print(f"Initial GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f}GB / {torch.cuda.memory_reserved()/1024**3:.2f}GB")
        torch.cuda.empty_cache()

    model.to(device)

    # Use deque for memory-efficient history storage (only keep recent values)
    history_size = min(num_epoch, 1000)  # Limit history to prevent memory issues
    train_loss_history = deque(maxlen=history_size)
    train_acc_history = deque(maxlen=history_size)
    val_loss_history = deque(maxlen=history_size)
    val_acc_history = deque(maxlen=history_size)

    # Improved optimizer with memory-efficient settings
    lr = 1e-3
    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4,
        eps=1e-8,  # Slightly larger eps for numerical stability in mixed precision
        amsgrad=False  # Disable amsgrad to save memory
    )

    # Learning rate scheduler
    lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=10,
        verbose=verbose,
        min_lr=1e-6
    )

    # Label smoothing for better generalization
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # Mixed precision training setup
    scaler = GradScaler() if use_amp else None

    # Early stopping variables
    best_val_loss = float('inf')
    best_val_acc = 0.0
    patience = 20
    patience_counter = 0
    best_model_path = '/tmp/best_model.pth'  # Save to disk instead of memory

    actual_epochs = 0

    try:
        for epoch in trange(num_epoch, desc='Training'):
            actual_epochs = epoch + 1

            # Memory cleanup every N epochs
            if epoch % memory_cleanup_freq == 0 and epoch > 0:
                gc.collect()
                if device == 'cuda':
                    torch.cuda.empty_cache()
                    if verbose:
                        tqdm.write(f"GPU memory after cleanup: {torch.cuda.memory_allocated()/1024**3:.2f}GB")

            # ----- Training Phase -----
            model.train()
            total_train_loss, total_train_correct, total_train_samples = 0.0, 0, 0

            # Reset gradients outside the loop
            optimizer.zero_grad()

            for batch_idx, (x, y, _) in enumerate(train_dl):
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

                # Gradient accumulation context
                with autocast(enabled=use_amp):
                    # Optional: Add noise augmentation (but less frequently to save memory)
                    if torch.rand(1).item() < 0.2:  # Reduced from 30% to 20%
                        noise = torch.randn_like(x) * 0.05
                        x = x + noise

                    logits = model(x)
                    loss = criterion(logits, y)

                    # Scale loss for gradient accumulation
                    loss = loss / accumulation_steps

                # Backward pass with mixed precision
                if use_amp:
                    scaler.scale(loss).backward()
                else:
                    loss.backward()

                # Update weights every accumulation_steps
                if (batch_idx + 1) % accumulation_steps == 0:
                    if use_amp:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                        optimizer.step()

                    optimizer.zero_grad()

                # Accumulate statistics (scale back the loss)
                with torch.no_grad():
                    total_train_loss += (loss.item() * accumulation_steps) * x.size(0)
                    total_train_correct += (logits.argmax(dim=1) == y).sum().item()
                    total_train_samples += x.size(0)

                # Clear intermediate tensors
                del x, y, logits, loss

            # Handle remaining gradients
            if (len(train_dl) % accumulation_steps) != 0:
                if use_amp:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                optimizer.zero_grad()

            epoch_train_loss = total_train_loss / total_train_samples
            epoch_train_acc = total_train_correct / total_train_samples

            train_loss_history.append(epoch_train_loss)
            train_acc_history.append(epoch_train_acc)

            # ----- Validation Phase -----
            model.eval()
            total_val_loss, total_val_correct, total_val_samples = 0.0, 0, 0

            with torch.no_grad():
                for x, y, _ in valid_dl:
                    x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

                    with autocast(enabled=use_amp):
                        logits = model(x)
                        loss = criterion(logits, y)

                    total_val_loss += loss.item() * x.size(0)
                    total_val_correct += (logits.argmax(dim=1) == y).sum().item()
                    total_val_samples += x.size(0)

                    # Clear tensors immediately
                    del x, y, logits, loss

            epoch_val_loss = total_val_loss / total_val_samples
            epoch_val_acc = total_val_correct / total_val_samples

            val_loss_history.append(epoch_val_loss)
            val_acc_history.append(epoch_val_acc)

            # Update learning rate
            lr_scheduler.step(epoch_val_loss)

            # Early stopping logic with disk-based model saving
            if epoch_val_loss < best_val_loss - 0.001:
                best_val_loss = epoch_val_loss
                best_val_acc = epoch_val_acc
                # Save best model to disk instead of keeping in memory
                torch.save(model.state_dict(), best_model_path)
                patience_counter = 0
            else:
                patience_counter += 1

            # Check if we should stop early
            if patience_counter >= patience:
                if verbose:
                    tqdm.write(f"\nEarly stopping triggered at epoch {epoch+1}")
                    tqdm.write(f"Best validation loss: {best_val_loss:.4f}, Best validation acc: {best_val_acc:.4f}")

                # Load best model from disk
                try:
                    model.load_state_dict(torch.load(best_model_path, map_location=device))
                except:
                    tqdm.write("Warning: Could not load best model state")
                break

            # Memory-conscious verbose output
            if verbose:
                show_progress = (
                    (epoch + 1) % 10 == 0 or
                    epoch < 10 or
                    epoch_val_acc > best_val_acc or
                    patience_counter == 0
                )

                if show_progress:
                    current_lr = optimizer.param_groups[0]['lr']
                    tqdm.write(f"Epoch {epoch+1:03d} | Train Loss: {epoch_train_loss:.4f}, Acc: {epoch_train_acc:.4f}")
                    tqdm.write(f"            | Val   Loss: {epoch_val_loss:.4f}, Acc: {epoch_val_acc:.4f}")
                    tqdm.write(f"            | LR: {current_lr:.6f}, Patience: {patience_counter}/{patience}")

                    if device == 'cuda':
                        tqdm.write(f"            | GPU Memory: {torch.cuda.memory_allocated()/1024**3:.2f}GB")

                    if epoch_val_acc > best_val_acc:
                        tqdm.write(f"            | *** New best validation accuracy! ***")

            # Force memory cleanup every few epochs
            if epoch % 5 == 0:
                gc.collect()

    except RuntimeError as e:
        if "out of memory" in str(e):
            print(f"\nGPU out of memory error at epoch {epoch+1}")
            print("Try reducing batch size, enabling gradient accumulation, or using mixed precision")
            print(f"Current GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f}GB")

            # Emergency cleanup
            gc.collect()
            torch.cuda.empty_cache()

        raise e

    finally:
        # Final cleanup
        gc.collect()
        if device == 'cuda':
            torch.cuda.empty_cache()

    # Convert deque to lists for compatibility
    train_history = {
        'train_loss': list(train_loss_history),
        'train_acc': list(train_acc_history),
        'val_loss': list(val_loss_history),
        'val_acc': list(val_acc_history),
        'best_val_loss': best_val_loss,
        'best_val_acc': best_val_acc,
        'epochs_trained': actual_epochs
    }

    if verbose:
        print(f"\nTraining completed!")
        print(f"Epochs trained: {actual_epochs}")
        print(f"Final validation accuracy: {val_acc_history[-1]:.4f}")
        print(f"Best validation accuracy: {best_val_acc:.4f}")
        if device == 'cuda':
            print(f"Final GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f}GB")

    return model, train_history


def get_memory_usage():
    """Utility function to check current GPU memory usage"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU Memory - Allocated: {allocated:.2f}GB, Reserved: {reserved:.2f}GB")
        return allocated, reserved
    return 0, 0


def optimize_dataloader_for_gpu(dataset, batch_size=512, num_workers=2):
    """
    Create memory-optimized dataloader for Colab Pro
    """
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,  # Reduced for Colab
        pin_memory=True,  # Faster GPU transfer
        persistent_workers=True if num_workers > 0 else False,
        prefetch_factor=2,  # Reduced prefetch
        drop_last=True  # Ensures consistent batch sizes
    )

In [17]:
# ============================================================================
# 2. IMPROVED MODEL ARCHITECTURE
# ============================================================================

def create_improved_model(n_labels, dropout_rate=0.4):
    """
    Create an improved CNN model with better regularization
    """
    class ImprovedCNN_Block(nn.Module):
        def __init__(self, in_channels, out_channels, dropout_rate=0.3):
            super().__init__()
            self.block = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(inplace=True),
                nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(inplace=True),
                nn.MaxPool1d(kernel_size=2),
                nn.Dropout(dropout_rate)
            )

        def forward(self, x):
            return self.block(x)

    class ImprovedCNN_NET(nn.Module):
        def __init__(self, n_labels, dropout_rate=0.4):
            super().__init__()

            self.backbone = nn.Sequential(
                ImprovedCNN_Block(2, 32, dropout_rate=0.2),
                ImprovedCNN_Block(32, 64, dropout_rate=0.3),
                ImprovedCNN_Block(64, 128, dropout_rate=0.3),
                nn.AdaptiveAvgPool1d(16)
            )

            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(128 * 16, 512),
                nn.BatchNorm1d(512),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout_rate),

                nn.Linear(512, 256),
                nn.BatchNorm1d(256),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout_rate * 0.75),

                nn.Linear(256, 128),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout_rate * 0.5),

                nn.Linear(128, n_labels)
            )

        def forward(self, x):
            x = self.backbone(x)
            return self.classifier(x)

    return ImprovedCNN_NET(n_labels, dropout_rate)

In [18]:
# ============================================================================
# 3. IMPROVED TESTING AND ANALYSIS FUNCTIONS
# ============================================================================

def test_model_with_improved_plots(model, device='cuda'):
    """
    Enhanced testing function with better memory management and error handling
    """
    model.eval()
    Y_pred_ = []  # Predictions
    Y_true_ = []  # Ground truth
    Z_snr_ = []   # SNR values

    # FIXED: Use actual test dataset instead of train
    test_dataset = RadioML18Dataset(mode='test')
    test_loader = DataLoader(dataset=test_dataset, batch_size=128, shuffle=False, drop_last=False)

    target_classes = test_dataset.target_modulations
    target_snrs = test_dataset.target_snrs
    modulation_classes = test_dataset.modulation_classes

    # Add debug
    print(f"Target modulations: {target_classes}")
    print(f"Target SNRs: {target_snrs}")
    print(f"Test dataset size: {len(test_dataset)}")

    # Initialize accuracy stats DataFrame
    accuracy_stats = pd.DataFrame(
        0.0,
        index=target_classes,
        columns=target_snrs.astype('str'))

    # Get predictions with tqdm progress bar
    test_progress = tqdm(test_loader, desc="Testing model", leave=True)

    with torch.no_grad():
        for batch_idx, (x, y, z) in enumerate(test_progress):
            # Move tensors to specified device
            x = x.to(device)
            y = y.to(device)
            z = z.to(device)

            # Get model predictions on device
            logits = model(x)
            y_pred = torch.argmax(logits, dim=-1)

            # Store results
            Y_pred_.append(y_pred.cpu())  # Move back to CPU for storage
            Y_true_.append(y.cpu())
            Z_snr_.append(z.cpu())

            # Update progress bar
            if batch_idx % 10 == 0:
                current_acc = (y_pred == y).float().mean().item()
                test_progress.set_postfix({"batch_acc": f"{current_acc:.3f}"})

            # Free up memory
            del x, y, z, logits, y_pred
            if device == 'cuda':
                torch.cuda.empty_cache()

    # Convert to numpy for easier processing
    Y_pred = torch.cat(Y_pred_).numpy()
    Y_true = torch.cat(Y_true_).numpy()
    Z_snr = torch.cat(Z_snr_).numpy()

    # Clear lists to free memory
    del Y_pred_, Y_true_, Z_snr_

    # Calculate overall accuracy
    correct_preds = (Y_pred == Y_true).sum()
    total_samples = len(Y_true)
    total_accuracy = round(correct_preds * 100 / total_samples, 2)
    print(f'Overall test accuracy: {total_accuracy}%')

    # Count samples for each modulation type
    mod_counts = {}
    for mod_idx, mod_name in enumerate(target_classes):
        count = np.sum(Y_true == mod_idx)
        mod_counts[mod_name] = count
        print(f"Modulation {mod_name}: {count} test samples")

    # Calculate accuracy per modulation and SNR with progress bar
    mod_snr_progress = tqdm(list(enumerate(target_classes)),
                           desc="Calculating per-modulation accuracies",
                           leave=True)

    for mod_idx, mod_name in mod_snr_progress:
        mod_snr_progress.set_postfix({"modulation": mod_name})
        for snr_idx, snr in enumerate(target_snrs):
            snr_str = str(snr)

            mask = (Y_true == mod_idx) & (Z_snr == snr)
            total_samples = mask.sum()
            if total_samples > 0:
                correct_samples = ((Y_pred == Y_true) & mask).sum()
                accuracy = (correct_samples * 100 / total_samples)
                accuracy_stats.loc[mod_name, snr_str] = round(accuracy, 2)
            else:
                accuracy_stats.loc[mod_name, snr_str] = np.nan
                print(f"Warning: no samples for {mod_name} at SNR = {snr}")

    return accuracy_stats, mod_counts, Y_true, Y_pred, target_classes

def plot_confusion_matrix(Y_true, Y_pred, target_classes, save_name='confusion_matrix'):
    """
    Plot confusion matrix for better understanding of misclassifications
    """
    cm = confusion_matrix(Y_true, Y_pred)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_classes,
                yticklabels=target_classes)
    plt.title('Confusion Matrix - Signal Modulation Classification')
    plt.xlabel('Predicted Modulation')
    plt.ylabel('True Modulation')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f'{save_name}.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Print classification report
    print("\nDetailed Classification Report:")
    print(classification_report(Y_true, Y_pred, target_names=target_classes))

def plot_improved_test_accuracy(model, device='cuda', save_prefix='model'):
    """
    Enhanced plotting function with confusion matrix and better analysis
    """
    accuracy_df, mod_counts, Y_true, Y_pred, target_classes = test_model_with_improved_plots(model, device)

    # 1. Plot confusion matrix first
    plot_confusion_matrix(Y_true, Y_pred, target_classes, f'{save_prefix}_confusion_matrix')

    # 2. Overall accuracy vs SNR plot
    plt.figure(figsize=(14, 8))

    accuracy_long = accuracy_df.reset_index().melt(
        id_vars=['index'],
        var_name='SNR',
        value_name='Accuracy'
    )
    accuracy_long.columns = ['Modulation', 'SNR', 'Accuracy']

    # Convert SNR to numeric for proper ordering
    accuracy_long['SNR_numeric'] = accuracy_long['SNR'].astype(int)
    accuracy_long = accuracy_long.sort_values('SNR_numeric')

    sns.lineplot(
        data=accuracy_long,
        x='SNR_numeric',
        y='Accuracy',
        hue='Modulation',
        marker='o',
        markersize=8,
        linewidth=2
    )

    # Highlight PSK modulations
    psk_mods = [mod for mod in accuracy_df.index if 'PSK' in mod]
    if psk_mods:
        print(f"Highlighting PSK modulations: {psk_mods}")
        for mod in psk_mods:
            mod_data = accuracy_long[accuracy_long['Modulation'] == mod]
            if not mod_data.empty:
                plt.plot(mod_data['SNR_numeric'], mod_data['Accuracy'],
                         linewidth=4,
                         linestyle='--',
                         marker='*',
                         markersize=12,
                         alpha=0.8)

    plt.title('Classification Accuracy vs SNR for Different Modulation Types', fontsize=16)
    plt.xlabel('Signal-to-Noise Ratio (dB)', fontsize=14)
    plt.ylabel('Accuracy (%)', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_all_modulations_accuracy.png', dpi=300, bbox_inches='tight')
    plt.show()

    # 3. Enhanced heatmap visualization
    plt.figure(figsize=(16, 8))

    # Reorder columns (SNRs) numerically
    snr_columns = sorted(accuracy_df.columns, key=int)
    accuracy_df_sorted = accuracy_df[snr_columns]

    # Create heatmap with better formatting
    mask = accuracy_df_sorted.isna()
    sns.heatmap(accuracy_df_sorted.astype(float),
                annot=True,
                cmap='RdYlGn',
                fmt='.1f',
                mask=mask,
                cbar_kws={'label': 'Accuracy (%)'},
                linewidths=0.5)

    plt.title('Classification Accuracy Heatmap by Modulation and SNR', fontsize=16)
    plt.xlabel('Signal-to-Noise Ratio (dB)', fontsize=14)
    plt.ylabel('Modulation Type', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_modulation_accuracy_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()

    # 4. Summary statistics
    print("\n" + "="*60)
    print("PERFORMANCE SUMMARY")
    print("="*60)

    overall_acc = np.nanmean(accuracy_df_sorted.values)
    print(f"Overall average accuracy: {overall_acc:.2f}%")

    # Best and worst performing modulations
    mod_avg_acc = accuracy_df_sorted.mean(axis=1).sort_values(ascending=False)
    print(f"\nBest performing modulation: {mod_avg_acc.index[0]} ({mod_avg_acc.iloc[0]:.2f}%)")
    print(f"Worst performing modulation: {mod_avg_acc.index[-1]} ({mod_avg_acc.iloc[-1]:.2f}%)")

    return accuracy_df_sorted

def plot_training_history(model_name, history):
    """
    Enhanced training history plotting with more details
    """
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

    epochs = range(1, len(history['train_loss']) + 1)

    # Loss plot
    ax1.plot(epochs, history['train_loss'], 'b-', label='Training Loss', linewidth=2)
    ax1.plot(epochs, history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
    ax1.set_title('Model Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Accuracy plot
    ax2.plot(epochs, [acc * 100 for acc in history['train_acc']], 'b-', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs, [acc * 100 for acc in history['val_acc']], 'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_title('Model Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # Overfitting analysis
    train_val_gap = [abs(t - v) * 100 for t, v in zip(history['train_acc'], history['val_acc'])]
    ax3.plot(epochs, train_val_gap, 'g-', linewidth=2)
    ax3.set_title('Train-Validation Gap (Overfitting Indicator)')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Accuracy Gap (%)')
    ax3.grid(True, alpha=0.3)

    # Show validation loss trend
    ax4.plot(epochs, history['val_loss'], 'purple', linewidth=2)
    ax4.set_title('Validation Loss Trend')
    ax4.set_xlabel('Epoch')
    ax4.set_ylabel('Validation Loss')
    ax4.grid(True, alpha=0.3)

    plt.suptitle(f'Training Analysis: {model_name}', fontsize=16)
    plt.tight_layout()
    plt.savefig(f'{model_name}_detailed_training_history.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Print training summary
    print(f"\nTraining Summary for {model_name}:")
    print(f"Epochs trained: {len(epochs)}")
    print(f"Final training accuracy: {history['train_acc'][-1]*100:.2f}%")
    print(f"Final validation accuracy: {history['val_acc'][-1]*100:.2f}%")
    if 'best_val_acc' in history:
        print(f"Best validation accuracy: {history['best_val_acc']*100:.2f}%")

def check_dataset_distribution(dataset_mode='test'):
    """
    Enhanced dataset analysis with better statistics
    """
    # Create dataset for analysis
    dataset = RadioML18Dataset(mode=dataset_mode)
    test_dl_analysis = DataLoader(dataset=dataset, batch_size=128, shuffle=False)

    # Analyze distribution
    mod_counts = {}
    snr_mod_counts = {}

    # Initialize counts for all modulations
    for mod in dataset.target_modulations:
        mod_counts[mod] = 0

    # Set up progress bar
    progress_bar = tqdm(range(len(dataset)), desc=f"Analyzing {dataset_mode} dataset", leave=True)

    # Count occurrences of each modulation
    for i in progress_bar:
        _, mod_idx, snr = dataset[i]
        mod = dataset.target_modulations[mod_idx]

        # Count by modulation
        mod_counts[mod] += 1

        # Count by modulation and SNR
        if snr not in snr_mod_counts:
            snr_mod_counts[snr] = {}
        if mod not in snr_mod_counts[snr]:
            snr_mod_counts[snr][mod] = 0
        snr_mod_counts[snr][mod] += 1

        # Update progress bar less frequently for performance
        if i % 1000 == 0:
            progress_bar.set_postfix({"current_mod": mod, "snr": snr})

    print(f"\n{dataset_mode.upper()} Dataset Distribution Analysis:")
    print("="*50)
    total_samples = sum(mod_counts.values())
    print(f"Total samples: {total_samples:,}")

    print("\nModulation distribution:")
    for mod, count in mod_counts.items():
        percentage = (count / total_samples) * 100
        print(f"  {mod}: {count:,} samples ({percentage:.1f}%)")

    # Check if dataset is balanced
    counts = list(mod_counts.values())
    is_balanced = max(counts) - min(counts) == 0
    print(f"\nDataset balance: {'Perfectly balanced' if is_balanced else 'Imbalanced'}")

    return mod_counts, snr_mod_counts

def improved_train_test_plots(model, model_name, verbose=True, device='cuda', num_epoch=200):
    """
    Complete training and testing pipeline with enhanced analysis
    """
    print("="*60)
    print(f"TRAINING AND EVALUATION PIPELINE: {model_name}")
    print("="*60)

    # First check the dataset distribution
    print("\n1. Analyzing dataset distribution...")
    mod_counts, snr_mod_counts = check_dataset_distribution('test')

    # Train the model
    print(f"\n2. Training {model_name}...")
    model, train_history = train_model(model, verbose=verbose, device=device, num_epoch=num_epoch)

    # Save the trained model
    torch.save(model.state_dict(), f'{model_name}_state_dict.pth')
    torch.save(model, f'{model_name}_full_model.pth')
    print(f"Model saved as {model_name}_state_dict.pth and {model_name}_full_model.pth")

    # Plot training history
    print("\n3. Plotting training history...")
    plot_training_history(model_name, train_history)

    # Test and analyze the model
    print("\n4. Testing model and generating analysis...")
    accuracy_results = plot_improved_test_accuracy(model, device, model_name)

    print("\n5. Analysis complete!")
    print("="*60)

    return model, train_history, accuracy_results

In [ ]:
improved_train_test_plots(
    model=create_improved_model(n_labels=19),
    model_name='ImprovedCNN_NET',
    device='cuda',
    verbose=True,
    num_epoch=200
)

TRAINING AND EVALUATION PIPELINE: ImprovedCNN_NET

1. Analyzing dataset distribution...


Analyzing test dataset: 100%|██████████| 1264640/1264640 [00:13<00:00, 95719.05it/s, current_mod=OQPSK, snr=30]



TEST Dataset Distribution Analysis:
Total samples: 1,264,640

Modulation distribution:
  OOK: 66,560 samples (5.3%)
  4ASK: 66,560 samples (5.3%)
  8ASK: 66,560 samples (5.3%)
  BPSK: 66,560 samples (5.3%)
  QPSK: 66,560 samples (5.3%)
  8PSK: 66,560 samples (5.3%)
  16PSK: 66,560 samples (5.3%)
  32PSK: 66,560 samples (5.3%)
  16APSK: 66,560 samples (5.3%)
  32APSK: 66,560 samples (5.3%)
  64APSK: 66,560 samples (5.3%)
  128APSK: 66,560 samples (5.3%)
  16QAM: 66,560 samples (5.3%)
  32QAM: 66,560 samples (5.3%)
  64QAM: 66,560 samples (5.3%)
  128QAM: 66,560 samples (5.3%)
  256QAM: 66,560 samples (5.3%)
  GMSK: 66,560 samples (5.3%)
  OQPSK: 66,560 samples (5.3%)

Dataset balance: Perfectly balanced

2. Training ImprovedCNN_NET...


/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
epochs:   0%|          | 1/200 [01:14<4:05:29, 74.02s/it]

Epoch 001 | Train Loss: 2.0114, Acc: 0.3852
            | Val   Loss: 1.8321, Acc: 0.4554
            | LR: 0.001000, Patience: 0/20


...

Epoch 014 | Train Loss: 1.7029, Acc: 0.5219
            | Val   Loss: 1.6845, Acc: 0.5234
            | LR: 0.001000, Patience: 3/20
            | *** New best validation accuracy! ***


epochs:  10%|█         | 20/200 [24:37<3:41:49, 73.94s/it]

Epoch 020 | Train Loss: 1.6886, Acc: 0.5284
            | Val   Loss: 1.6925, Acc: 0.5222
            | LR: 0.001000, Patience: 9/20


epochs:  12%|█▏        | 23/200 [28:20<3:38:38, 74.11s/it]

Epoch 023 | Train Loss: 1.6720, Acc: 0.5354
            | Val   Loss: 1.7025, Acc: 0.5248
            | LR: 0.000500, Patience: 12/20
            | *** New best validation accuracy! ***


In [ ]:
improved_train_test_plots(
    model=CNN_NET(n_labels),
    model_name = 'CNN_NET',
    device = 'cuda',
    verbose = True,
    num_epoch=100
)

In [ ]:
class DDrCNN2D(nn.Module):
    """
    2D-Conv version optimized for radio signal classification:
    - Input is (batch, 1, 2, 1024)
    - First conv uses a (2×3) kernel to collapse height → 1
    - Then we have a stack of (1×3) time-only convs / pools
    - Enhanced with better regularization
    """
    def __init__(self, num_classes=6, input_shape=(2, 1024), dropout_rate=0.3):
        super(DDrCNN2D, self).__init__()
        H, W = input_shape  # H=2, W=1024

        self.backbone = nn.Sequential(
            # Conv #1: Collapse I/Q channels (H=2→1) and learn I/Q combination
            nn.Conv2d(1, 32, kernel_size=(2,3), padding=(0,1)),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout_rate * 0.5),  # Light dropout early
            nn.MaxPool2d((1,2)),    # now (batch, 32, 1, W/2)

            # Conv #2: Time-domain feature extraction
            nn.Conv2d(32, 32, kernel_size=(1,3), padding=(0,1)),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=(1,3), padding=(0,1)),  # Additional conv
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout_rate * 0.7),
            nn.MaxPool2d((1,2)),

            # Conv #3: Deeper feature extraction
            nn.Conv2d(32, 64, kernel_size=(1,3), padding=(0,1)),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=(1,3), padding=(0,1)),  # Additional conv
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout_rate),
            nn.MaxPool2d((1,2)),

            # Conv #4: High-level feature extraction
            nn.Conv2d(64, 128, kernel_size=(1,3), padding=(0,1)),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=(1,3), padding=(0,1)),  # Additional conv
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout_rate),
            nn.MaxPool2d((1,2)),

            # Global average pooling to reduce overfitting
            nn.AdaptiveAvgPool2d((1, 8)),  # Fixed size output
        )

        # Calculate flattened dimensions
        flat_dim = 128 * 1 * 8  # From AdaptiveAvgPool2d output

        # Enhanced classification head
        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(flat_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate * 0.75),

            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate * 0.5),

            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        # x is (batch, 2, 1024) → make it 'grayscale image' (batch, 1, 2, 1024)
        x = x.unsqueeze(1)
        x = self.backbone(x)
        x = self.classifier(x)
        return x

In [ ]:
improved_train_test_plots(
    model=DDrCNN2D(n_labels),
    model_name = 'DDrCNN2D',
    device = 'cuda',
    verbose = True,
    num_epoch=100
)

In [ ]:
class EnhancedDDrCNN2D(nn.Module):
    """
    Enhanced version with residual connections and attention
    """
    def __init__(self, num_classes=6, input_shape=(2, 1024), dropout_rate=0.3):
        super(EnhancedDDrCNN2D, self).__init__()
        H, W = input_shape

        # I/Q combination layer
        self.iq_combiner = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(2,3), padding=(0,1)),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout_rate * 0.3),
        )

        # Feature extraction blocks with residual connections
        self.block1 = self._make_block(32, 32, dropout_rate)
        self.block2 = self._make_block(32, 64, dropout_rate)
        self.block3 = self._make_block(64, 128, dropout_rate)

        # Attention mechanism
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(128, 64, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 1),
            nn.Sigmoid()
        )

        # Final pooling
        self.final_pool = nn.AdaptiveAvgPool2d((1, 8))

        # Classification head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate * 0.75),

            nn.Linear(128, num_classes),
        )

    def _make_block(self, in_channels, out_channels, dropout_rate):
        """Create a residual-like block"""
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(1,3), padding=(0,1)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=(1,3), padding=(0,1)),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout_rate),
            nn.MaxPool2d((1,2)),
        )

    def forward(self, x):
        # x is (batch, 2, 1024) → make it 'grayscale image' (batch, 1, 2, 1024)
        x = x.unsqueeze(1)

        # I/Q combination
        x = self.iq_combiner(x)

        # Feature extraction
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)

        # Apply attention
        attention_weights = self.attention(x)
        x = x * attention_weights

        # Final processing
        x = self.final_pool(x)
        x = self.classifier(x)
        return x

# FIXED CREATION FUNCTIONS
def create_ddrcnn2d_model(num_classes=6, dropout_rate=0.3, enhanced=False):
    """Create DDrCNN2D model with correct parameter order"""
    if enhanced:
        return EnhancedDDrCNN2D(num_classes=num_classes, dropout_rate=dropout_rate)
    else:
        return DDrCNN2D(num_classes=num_classes, dropout_rate=dropout_rate)

In [ ]:
improved_train_test_plots(
    model=create_ddrcnn2d_model(n_labels),
    model_name = 'DDrCNN2D_improve',
    device = 'cuda',
    verbose = True,
    num_epoch=100
)

In [ ]:
def compare_all_models(num_epoch=150, device='cuda', quick_test=False):
    """
    Compare all 4 models comprehensively

    Args:
        num_epoch: Number of epochs to train each model
        device: Device to use for training
        quick_test: If True, trains for fewer epochs for quick testing

    Returns:
        complete_results: Dictionary with all results
    """

    if quick_test:
        num_epoch = 50
        print("🚀 QUICK TEST MODE - Training for 50 epochs each")

    print("="*80)
    print("🏆 COMPREHENSIVE 4-MODEL COMPARISON")
    print("="*80)
    print(f"Training each model for up to {num_epoch} epochs (with early stopping)")
    print(f"Device: {device}")
    print("="*80)

    # Define all 4 models to compare
    models_to_test = {
        '1_Original_CNN': CNN_NET(n_labels),
        '2_Improved_CNN': create_improved_model(n_labels=6, dropout_rate=0.4),
        '3_DDrCNN2D_Basic': create_ddrcnn2d_model(num_classes=6, dropout_rate=0.3),
        '4_DDrCNN2D_Enhanced': create_ddrcnn2d_model(num_classes=6, dropout_rate=0.3, enhanced=True),
    }

    results = {}
    training_times = {}

    # Train each model
    for i, (model_name, model) in enumerate(models_to_test.items(), 1):
        print(f"\n{'='*20} MODEL {i}/4: {model_name} {'='*20}")

        start_time = time.time()

        # Train and test the model
        trained_model, history, accuracy_df = improved_train_test_plots(
            model=model,
            model_name=model_name,
            device=device,
            verbose=True,
            num_epoch=num_epoch
        )

        end_time = time.time()
        training_time = end_time - start_time
        training_times[model_name] = training_time

        # Calculate additional metrics
        final_val_acc = history['val_acc'][-1].item() * 100
        best_val_acc = history.get('best_val_acc', max([acc.item() for acc in history['val_acc']])) * 100
        overall_test_acc = accuracy_df.mean().mean()
        epochs_trained = history.get('epochs_trained', len(history['val_acc']))

        # Per-modulation performance
        modulation_performance = accuracy_df.mean(axis=1).to_dict()  # Average across SNRs

        # Store results
        results[model_name] = {
            'model': trained_model,
            'history': history,
            'accuracy_df': accuracy_df,
            'final_val_acc': final_val_acc,
            'best_val_acc': best_val_acc,
            'overall_test_acc': overall_test_acc,
            'epochs_trained': epochs_trained,
            'training_time': training_time,
            'modulation_performance': modulation_performance,
        }

        print(f"\n✅ {model_name} COMPLETED:")
        print(f"   🎯 Final Val Acc: {final_val_acc:.2f}%")
        print(f"   🏆 Best Val Acc:  {best_val_acc:.2f}%")
        print(f"   📊 Test Acc:      {overall_test_acc:.2f}%")
        print(f"   ⏱️  Training Time: {training_time/60:.1f} minutes")
        print(f"   📈 Epochs:        {epochs_trained}")

    # Create comprehensive comparison
    create_comprehensive_comparison(results, training_times)

    return results

def create_comprehensive_comparison(results, training_times):
    """Create detailed comparison visualizations and analysis"""

    model_names = list(results.keys())
    n_models = len(model_names)

    # Clean model names for display (remove numbers)
    display_names = [name.split('_', 1)[1] if '_' in name else name for name in model_names]

    # Create mega comparison figure
    fig = plt.figure(figsize=(20, 16))

    # 1. Training Curves (2x2 subplot in top half)
    # Validation Accuracy
    ax1 = plt.subplot(3, 3, 1)
    for i, model_name in enumerate(model_names):
        history = results[model_name]['history']
        epochs = range(1, len(history['val_acc']) + 1)
        val_acc = [acc.item() * 100 for acc in history['val_acc']]
        ax1.plot(epochs, val_acc, label=display_names[i], linewidth=2.5, marker='o', markersize=2)

    ax1.set_title('🎯 Validation Accuracy Evolution', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Validation Accuracy (%)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Validation Loss
    ax2 = plt.subplot(3, 3, 2)
    for i, model_name in enumerate(model_names):
        history = results[model_name]['history']
        epochs = range(1, len(history['val_loss']) + 1)
        val_loss = [loss.item() for loss in history['val_loss']]
        ax2.plot(epochs, val_loss, label=display_names[i], linewidth=2.5, marker='o', markersize=2)

    ax2.set_title('📉 Validation Loss Evolution', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Validation Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    # 2. Performance Comparison
    ax3 = plt.subplot(3, 3, 3)
    final_accs = [results[name]['final_val_acc'] for name in model_names]
    best_accs = [results[name]['best_val_acc'] for name in model_names]
    test_accs = [results[name]['overall_test_acc'] for name in model_names]

    x = np.arange(len(display_names))
    width = 0.25

    bars1 = ax3.bar(x - width, final_accs, width, label='Final Val', alpha=0.8, color='skyblue')
    bars2 = ax3.bar(x, best_accs, width, label='Best Val', alpha=0.8, color='lightgreen')
    bars3 = ax3.bar(x + width, test_accs, width, label='Test', alpha=0.8, color='lightcoral')

    # Add value labels on bars
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            ax3.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    f'{height:.1f}%', ha='center', va='bottom', fontsize=8)

    ax3.set_title('🏆 Performance Comparison', fontsize=14, fontweight='bold')
    ax3.set_ylabel('Accuracy (%)')
    ax3.set_xlabel('Model')
    ax3.set_xticks(x)
    ax3.set_xticklabels(display_names, rotation=45, ha='right')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # 3. Training Efficiency
    ax4 = plt.subplot(3, 3, 4)
    epochs_trained = [results[name]['epochs_trained'] for name in model_names]
    times = [training_times[name]/60 for name in model_names]  # Convert to minutes

    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
    bars = ax4.bar(display_names, epochs_trained, color=colors[:len(display_names)], alpha=0.8)

    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{int(height)}', ha='center', va='bottom', fontsize=10)

    ax4.set_title('⏱️ Training Efficiency (Epochs)', fontsize=14, fontweight='bold')
    ax4.set_ylabel('Epochs to Convergence')
    ax4.set_xlabel('Model')
    ax4.set_xticklabels(display_names, rotation=45, ha='right')
    ax4.grid(True, alpha=0.3)

    # 4. Training Time
    ax5 = plt.subplot(3, 3, 5)
    bars = ax5.bar(display_names, times, color=colors[:len(display_names)], alpha=0.8)

    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax5.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{height:.1f}m', ha='center', va='bottom', fontsize=10)

    ax5.set_title('⏰ Training Time', fontsize=14, fontweight='bold')
    ax5.set_ylabel('Training Time (minutes)')
    ax5.set_xlabel('Model')
    ax5.set_xticklabels(display_names, rotation=45, ha='right')
    ax5.grid(True, alpha=0.3)

    # 5. Per-Modulation Performance Heatmap
    ax6 = plt.subplot(3, 3, (6, 9))  # Span multiple subplot positions

    # Combine all accuracy dataframes
    modulation_data = []
    modulation_names = None

    for model_name in model_names:
        acc_df = results[model_name]['accuracy_df']
        avg_per_mod = acc_df.mean(axis=1)  # Average across SNRs
        modulation_data.append(avg_per_mod.values)
        if modulation_names is None:
            modulation_names = acc_df.index.tolist()

    modulation_array = np.array(modulation_data)

    # Create heatmap
    im = ax6.imshow(modulation_array, cmap='RdYlGn', aspect='auto', vmin=30, vmax=100)

    # Set ticks and labels
    ax6.set_xticks(np.arange(len(modulation_names)))
    ax6.set_yticks(np.arange(len(display_names)))
    ax6.set_xticklabels(modulation_names)
    ax6.set_yticklabels(display_names)

    # Add text annotations
    for i in range(len(display_names)):
        for j in range(len(modulation_names)):
            text = ax6.text(j, i, f'{modulation_array[i, j]:.1f}%',
                           ha="center", va="center", color="black", fontweight='bold')

    ax6.set_title('📊 Per-Modulation Performance Heatmap\n(Average Across All SNRs)',
                  fontsize=14, fontweight='bold')
    ax6.set_xlabel('Modulation Type')
    ax6.set_ylabel('Model')

    # Add colorbar
    cbar = plt.colorbar(im, ax=ax6, fraction=0.046, pad=0.04)
    cbar.set_label('Accuracy (%)', rotation=270, labelpad=15)

    # 6. Improvement Analysis
    ax7 = plt.subplot(3, 3, 7)
    baseline_acc = results[model_names[0]]['best_val_acc']  # Original CNN as baseline
    improvements = [(results[name]['best_val_acc'] - baseline_acc) for name in model_names[1:]]
    improved_names = display_names[1:]

    colors_imp = ['green' if imp > 0 else 'red' for imp in improvements]
    bars = ax7.bar(improved_names, improvements, color=colors_imp, alpha=0.8)

    for bar, imp in zip(bars, improvements):
        height = bar.get_height()
        ax7.text(bar.get_x() + bar.get_width()/2., height + (0.2 if height > 0 else -0.2),
                f'{imp:+.1f}%', ha='center', va='bottom' if height > 0 else 'top', fontsize=10)

    ax7.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    ax7.set_title('📈 Improvement vs Original CNN', fontsize=14, fontweight='bold')
    ax7.set_ylabel('Accuracy Improvement (%)')
    ax7.set_xlabel('Model')
    ax7.set_xticklabels(improved_names, rotation=45, ha='right')
    ax7.grid(True, alpha=0.3)

    # 7. Model Complexity (Parameter Count)
    ax8 = plt.subplot(3, 3, 8)
    param_counts = []
    for model_name in model_names:
        model = results[model_name]['model']
        total_params = sum(p.numel() for p in model.parameters())
        param_counts.append(total_params / 1000)  # Convert to thousands

    bars = ax8.bar(display_names, param_counts, color=colors[:len(display_names)], alpha=0.8)

    for bar, params in zip(bars, param_counts):
        height = bar.get_height()
        ax8.text(bar.get_x() + bar.get_width()/2., height + 5,
                f'{params:.0f}K', ha='center', va='bottom', fontsize=9)

    ax8.set_title('🔧 Model Complexity', fontsize=14, fontweight='bold')
    ax8.set_ylabel('Parameters (Thousands)')
    ax8.set_xlabel('Model')
    ax8.set_xticklabels(display_names, rotation=45, ha='right')
    ax8.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('comprehensive_4_model_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Print comprehensive summary
    print_final_comparison_report(results, training_times, display_names)

def print_final_comparison_report(results, training_times, display_names):
    """Print detailed comparison report"""
    model_names = list(results.keys())

    print("\n" + "="*100)
    print("🏆 FINAL COMPREHENSIVE COMPARISON REPORT")
    print("="*100)

    # Summary table
    print(f"{'Model':<20} {'Best Val':<10} {'Test Acc':<10} {'Epochs':<8} {'Time(min)':<10} {'Params(K)':<10}")
    print("-" * 100)

    for i, model_name in enumerate(model_names):
        r = results[model_name]
        model = r['model']
        total_params = sum(p.numel() for p in model.parameters()) / 1000
        time_min = training_times[model_name] / 60

        print(f"{display_names[i]:<20} {r['best_val_acc']:<10.2f} {r['overall_test_acc']:<10.2f} "
              f"{r['epochs_trained']:<8} {time_min:<10.1f} {total_params:<10.0f}")

    # Find winners in each category
    best_accuracy = max(model_names, key=lambda x: results[x]['best_val_acc'])
    fastest_training = min(model_names, key=lambda x: training_times[x])
    most_efficient = min(model_names, key=lambda x: results[x]['epochs_trained'])
    smallest_model = min(model_names, key=lambda x: sum(p.numel() for p in results[x]['model'].parameters()))

    print(f"\n🎯 CATEGORY WINNERS:")
    print(f"🏆 Best Accuracy:     {display_names[model_names.index(best_accuracy)]} ({results[best_accuracy]['best_val_acc']:.2f}%)")
    print(f"⚡ Fastest Training:   {display_names[model_names.index(fastest_training)]} ({training_times[fastest_training]/60:.1f} min)")
    print(f"📈 Most Efficient:    {display_names[model_names.index(most_efficient)]} ({results[most_efficient]['epochs_trained']} epochs)")
    print(f"🗜️  Smallest Model:    {display_names[model_names.index(smallest_model)]} ({sum(p.numel() for p in results[smallest_model]['model'].parameters())/1000:.0f}K params)")

    # Overall champion (weighted score)
    print(f"\n🏆 OVERALL ANALYSIS:")
    baseline_acc = results[model_names[0]]['best_val_acc']

    for i, model_name in enumerate(model_names[1:], 1):
        improvement = results[model_name]['best_val_acc'] - baseline_acc
        print(f"📊 {display_names[i]}: {improvement:+.2f}% improvement over Original CNN")

    # Recommendations
    print(f"\n💡 RECOMMENDATIONS:")
    if results[best_accuracy]['best_val_acc'] > 75:
        print(f"✅ {display_names[model_names.index(best_accuracy)]} achieved excellent performance (>75% accuracy)")

    best_improved = max(model_names[1:], key=lambda x: results[x]['best_val_acc'])
    print(f"🚀 Best upgrade from Original CNN: {display_names[model_names.index(best_improved)]}")

    print("="*100)

# READY-TO-RUN FUNCTIONS

def quick_model_comparison():
    """Quick 50-epoch comparison for testing"""
    return compare_all_models(num_epoch=50, quick_test=True)

def full_model_comparison():
    """Full comparison with up to 150 epochs each"""
    return compare_all_models(num_epoch=150, quick_test=False)

In [ ]:
results = full_model_comparison()